In [1]:
import pandas as pd

data = pd.read_csv("../data/reviews.csv")

data.sample(5)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
455829,455830,B007HERNSO,A3EYP9APJQF5OQ,Jennifer S.,1,1,5,1346544000,Exceptional Cat Food!,"Oh, this is some good cat food!<br />Have you ..."
424105,424106,B001EQ51JE,AP6JW5EAWRZ5Q,"Charlene C. Sanders ""spaceplanner""",2,3,5,1167436800,Love this Raisin Bran!!,I used to drive 45 miles to purchase this cere...
299133,299134,B001TZJ2MW,AIMVGS5Q10B6B,Betty Little-royer,8,8,5,1306022400,great vinegar,This product is a favorite of mine - I have be...
370475,370476,B004PEN59U,A2NOGPJMDPJC9M,jennief23,1,1,5,1298937600,Delicious!!!,Our family loves fruit now we just have a diff...
341702,341703,B001RVFEP2,AIN0ZUU09525J,"NotASheep ""Booklover""",2,5,1,1303862400,"To me, ""Original"" flavor just nasty tasting.","Unfortunately, I somehow stumbled upon this pr..."


Como funciona essa timestamp

O campo Time é um Unix Timestamp (ou Epoch time).

O que isso significa?

É o número de segundos desde 01/01/1970 (UTC).



| Coluna                   | Significado                           |
| ------------------------ | ------------------------------------- |
| `HelpfulnessNumerator`   | Quantas pessoas acharam a review útil |
| `HelpfulnessDenominator` | Quantas pessoas votaram na review     |

3 / 5
3 pessoas acharam util, 5 votaram.


In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568428 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


temos alguns valores nulos em summary -> Podemos inferir um sumário para eles e isso até servirá como teste de inferencia.

In [3]:
data["review_date"] = pd.to_datetime(data["Time"], unit="s")

In [4]:
data.drop(columns=["Id","ProfileName","Time"], inplace=True )

In [5]:
data["helpfulness_score"] = (
    data["HelpfulnessNumerator"] /
    data["HelpfulnessDenominator"]
)

In [6]:
data.head(5)

,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score
0,B001E4KFG0,A3SGXH7AUHU8GW,1,1,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,2011-04-27,1.0
1,B00813GRG4,A1D87F6ZCVE5NK,0,0,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,2012-09-07,NaN
2,B000LQOCH0,ABXLMWJIXXAIN,1,1,4,"""Delight"" says it all",This is a confection that has been around a fe...,2008-08-18,1.0
3,B000UA0QIQ,A395BORC6FGVXV,3,3,2,Cough Medicine,If you are looking for the secret ingredient i...,2011-06-13,1.0
4,B006K2ZZ7K,A1UQRSCLF8GW1T,0,0,5,Great taffy,Great taffy at a great price. There was a wid...,2012-10-21,NaN


In [7]:
data.helpfulness_score.info()

<class 'pandas.core.series.Series'>
RangeIndex: 568454 entries, 0 to 568453
Series name: helpfulness_score
Non-Null Count   Dtype  
--------------   -----  
298402 non-null  float64
dtypes: float64(1)
memory usage: 4.3 MB


Podemos perceber agora que tem apenas 298402 valores não nulos, isso vem do fato de que HelpfulnessNumerator e HelpfulnessDenominator possuiam zeros, podemos descartar estes registros pois eles possuem muito pouca confiabilidade.

In [8]:
data[data["HelpfulnessDenominator"] ==0].shape

(270052, 9)

In [9]:
data = data[data["HelpfulnessDenominator"] !=0]

In [10]:
data.shape

(298402, 9)

Registros sem avaliações, devemos apagar

In [11]:
data.nunique()

ProductId                  49316
UserId                    149991
HelpfulnessNumerator         231
HelpfulnessDenominator       233
Score                          5
Summary                   168010
Text                      209488
review_date                 3130
helpfulness_score            951
dtype: int64

In [12]:
data.Score.min()

np.int64(1)

Temos 298402 registros, 49316 produtos distintos, 149991 usuários, 231 valores distintos de HelpfulnessNumerator e 233 no denominator, score vai de 1 a 5, summary 168010 (repetidos?) Text 209488 (repetidos?)

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298402 entries, 0 to 568452
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ProductId               298402 non-null  object        
 1   UserId                  298402 non-null  object        
 2   HelpfulnessNumerator    298402 non-null  int64         
 3   HelpfulnessDenominator  298402 non-null  int64         
 4   Score                   298402 non-null  int64         
 5   Summary                 298376 non-null  object        
 6   Text                    298402 non-null  object        
 7   review_date             298402 non-null  datetime64[ns]
 8   helpfulness_score       298402 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 22.8+ MB


In [14]:
summary_to_infer = data[data["Summary"].isna()]
summary_to_infer.head()


,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score
33958,B00412W76S,A3TJPSWY2HE4BS,1,24,2,NaN,I only used two maybe three tea bags and got p...,2007-03-08,0.041667
40548,B00020HHRW,A3TJPSWY2HE4BS,1,24,2,NaN,I only used two maybe three tea bags and got p...,2007-03-08,0.041667
101106,B0014B0HWK,A3TJPSWY2HE4BS,1,24,2,NaN,I only used two maybe three tea bags and got p...,2007-03-08,0.041667
102979,B000FVDWU4,A3TJPSWY2HE4BS,1,24,2,NaN,I only used two maybe three tea bags and got p...,2007-03-08,0.041667
117515,B0016B7Z32,A3TJPSWY2HE4BS,1,24,2,NaN,I only used two maybe three tea bags and got p...,2007-03-08,0.041667


In [15]:
data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298402 entries, 0 to 568452
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ProductId               298402 non-null  object        
 1   UserId                  298402 non-null  object        
 2   HelpfulnessNumerator    298402 non-null  int64         
 3   HelpfulnessDenominator  298402 non-null  int64         
 4   Score                   298402 non-null  int64         
 5   Summary                 298376 non-null  object        
 6   Text                    298402 non-null  object        
 7   review_date             298402 non-null  datetime64[ns]
 8   helpfulness_score       298402 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 22.8+ MB


Cleaning the text : 

In [16]:
data["review_length"] = data["Text"].str.split().str.len()

data["review_length"].describe()


count    298402.000000
mean         89.660401
std          91.337398
min           3.000000
25%          36.000000
50%          63.000000
75%         109.000000
max        3432.000000
Name: review_length, dtype: float64

In [17]:

short_reviews = data[data["review_length"] < 10]

short_reviews[["Text", "review_length"]].head(20)


,Text,review_length
15560,Same price as Dr. Foster & Smith.,7
44455,"Yummy, if you like noodles I recomend these.",8
49025,"<span class=""tiny""> Length:: 1:02 Mins<br /><b...",7
65997,"<span class=""tiny""> Length:: 0:42 Mins<br /><b...",7
70602,Tastes good. Fruit juice sweetened. Excellent ...,7
80655,Great gift. Fast delivery. Quality product.,6
94052,Same price as Dr. Foster & Smith.,7
106960,Tastes great and the price was good.,7
107456,"<span class=""tiny""> Length:: 4:34 Mins<br /><b...",7
107554,Tastes great and the price was good.,7


In [18]:
mask_span_class = data["Text"].str.contains(
    r"<span\s+class",
    regex=True,
    case=False,
    na=False
)

data[mask_span_class][["Text"]].head(10)


,Text
158,"<span class=""tiny""> Length:: 0:26 Mins<br /><b..."
1460,"<span class=""tiny""> Length:: 1:38 Mins<br /><b..."
5487,"<span class=""tiny""> Length:: 0:32 Mins<br /><b..."
11803,"<span class=""tiny""> Length:: 0:51 Mins<br /><b..."
14064,"<span class=""tiny""> Length:: 1:39 Mins<br /><b..."
14157,"<span class=""tiny""> Length:: 1:15 Mins<br /><b..."
14197,"<span class=""tiny""> Length:: 1:10 Mins<br /><b..."
15828,"<span class=""tiny""> Length:: 1:30 Mins<br /><b..."
16053,"<span class=""tiny""> Length:: 2:09 Mins<br /><b..."
19164,"<span class=""tiny""> Length:: 0:39 Mins<br /><b..."


In [19]:
mask_span_class.mean() * 100

np.float64(0.10254622958291164)

In [20]:
import re

def remove_span_tags(text):
    return re.sub(r"</?span[^>]*>", " ", text)

data["Text"] = data["Text"].apply(remove_span_tags)


In [21]:
dup_mask = data.duplicated(subset=["Text"], keep=False)

dup_mask.sum()


np.int64(117723)

In [22]:
dup_ratio = dup_mask.mean()
dup_ratio * 100


np.float64(39.45114308885329)

In [23]:
data = (
    data
    .sort_values(
        ["HelpfulnessDenominator", "helpfulness_score"],
        ascending=False
    )
    .drop_duplicates(subset=["Text"], keep="first")
)


Antes de remover duplicatas, os dados foram ordenados pelo número total de votos
(HelpfulnessDenominator) e pelo helpfulness_score, garantindo que reviews mais
confiáveis e com maior engajamento apareçam primeiro.

Em seguida, duplicatas de texto foram removidas mantendo apenas a primeira
ocorrência, que corresponde à review com maior evidência social. Dessa forma,
evita-se viés causado por textos repetidos, reduz-se ruído no dataset e preserva-se
a versão mais representativa de cada review para as etapas de NLP.


In [24]:
data.duplicated(subset=["Text"]).sum()


np.int64(0)

In [25]:
data.shape

(209488, 10)

In [26]:
data["Text"] = (
    data["Text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

data.head()

,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score,review_length
207712,B00012182G,A1JUGIQDY6UYSM,844,923,3,Whole Rabbit - NOT!,"I ordered one of these Fresh ""Whole"" Rabbits, ...",2009-09-08,0.914410,111
190733,B000FI4O90,A1GQGYROVZVW49,866,878,5,Works as Advertised - Classy Product,see update at end of review<br /><br />*******...,2006-11-28,0.986333,1223
566779,B001PQTYN2,A1QB2Y8GSME58Y,808,815,5,sauce not for mortals,I purchased a burrito from a small shop a few ...,2009-12-14,0.991411,346
235722,B001F10XUU,A39V22BIBUMMB3,580,593,1,Lost in Translation: Truth,"This product is called ""Hunmatsu-RyokuCha,"" in...",2011-07-02,0.978078,770
222937,B000UUWECC,AU3GYRAKBUAEU,491,569,3,"not bad stuff, but I have serious questions",Coconut water is the liquid inside an unopened...,2008-06-01,0.862917,708


In [27]:
data.shape

(209488, 10)

In [28]:
data["total_reviews"] = (
    data
    .groupby("ProductId")["Score"]
    .transform("count")
)

data["total_reviews"].describe()


count    209488.000000
mean         31.745503
std          54.980779
min           1.000000
25%           4.000000
50%          11.000000
75%          32.000000
max         451.000000
Name: total_reviews, dtype: float64

In [29]:
data = data[data["total_reviews"] >= 32]


In [31]:
data.shape

(52405, 11)

Products with fewer than 32 reviews were removed. According to the
distribution of total_reviews, this threshold removes the majority of the data, but preserves the most reviewed ones, fiz isso pois tinhamos muitos dados e demora muito para fazer o preprocessing de nlp

In [30]:
data.to_csv("../data/data_cleaned.csv")